# Example experiment on the Microscope Agnostic Controller

The general flow a workflow follows on top of the controller. It talks only to
the controller - never a vendor API - so the same notebook runs against any
microscope that has a driver.

1. connect - pick an instrument and set the reference frame
2. select states - capture a configuration per scan mode
3. get positions - the positions to visit
4. acquire - set a mode, then for each position: set_xyz, acquire (captures + saves)

## Setup

Register the bundled **mock driver** so the flow runs with no hardware. Run this
notebook with its own folder as the working directory.

In [ ]:
import sys
from pathlib import Path

here = Path.cwd()  # .../microscopes/microscope_agnostic_controller
sys.path.insert(0, str(here.parents[0]))  # microscopes/ source root
sys.path.insert(0, str(here / "tests"))  # the mock driver

import mock_driver  # noqa: E402

mock_driver.register_mock()

## 1. Connect

`get_instruments()` lists what you can connect to, each with its objective and
actuator options. Pick one and `set_instrument()` opens the session and fixes the
reference frame (objective + actuators) in a single step. From here, `mac` is that
microscope.

In [ ]:
import microscope_agnostic_controller as mac

mac.get_instruments()

In [ ]:
instrument = mac.get_instruments()[0]
mac.set_instrument(
    instrument,
    reference_actuators={"x": "motoric", "y": "motoric", "z": "motoric"},
    reference_objective="10x",
)

## 2. Select states

A **state** is a snapshot of the instrument you can reapply later: an `immutable`
fingerprint plus a `mutable` part. It is opaque to the controller - only the
driver interprets it. We capture two modes, one cell each.

In [ ]:
# a gentle overview scan
prescan = mac.get_state()
prescan["mutable"]["laser_power"] = 2.0
prescan

In [ ]:
# a higher-power detailed scan
target = mac.get_state()
target["mutable"]["laser_power"] = 8.0
target["mutable"]["gain"] = 2.0
target

## 3. Get the positions

`get_context()` returns whatever extra context the driver provides; here that
includes the positions captured at connect.

In [ ]:
positions = mac.get_context()["initial_positions"]
positions

## 4. Acquire

Reapply a state, then run every position. `set_xyz` moves in the motoric frame
(`with_actuators` picks the actuator per axis - here the piezo for fine Z).
`acquire` captures **and** saves in one call: `acquisition_type` is the scan
kind, `position_label` names the output, and `options` (from
`get_acquisition_options()`) carries acquisition + saving settings. Re-run with
`mac.set_state(target)` for the target scan.

In [ ]:
mac.set_state(prescan)

In [ ]:
for i, pos in enumerate(positions):
    mac.set_xyz(pos["x"], pos["y"], pos["z"], with_actuators={"z": "piezo"})
    record = mac.acquire(acquisition_type="prescan", position_label=f"pos_{i:03d}")
    print(record)

## Done

Close the session. The mock has nothing to release, but a real driver may need
the teardown.

---

*Author: Thom de Hoog — Center for Microscopy and Image Analysis (ZMB),
University of Zurich (thom.dehoog@zmb.uzh.ch, thomdehoog@gmail.com).*

In [ ]:
mac.disconnect()